# Fine-Tune Whisper For Multilingual ASR with 🤗 Transformers

In this Colab, we present a step-by-step guide on how to fine-tune Whisper
for any multilingual ASR dataset using Hugging Face 🤗 Transformers. This is a
more "hands-on" version of the accompanying [blog post](https://huggingface.co/blog/fine-tune-whisper).
For a more in-depth explanation of Whisper, the Common Voice dataset and the theory behind fine-tuning, the reader is advised to refer to the blog post.

## Introduction

Whisper is a pre-trained model for automatic speech recognition (ASR)
published in [September 2022](https://openai.com/blog/whisper/) by the authors
Alec Radford et al. from OpenAI. Unlike many of its predecessors, such as
[Wav2Vec 2.0](https://arxiv.org/abs/2006.11477), which are pre-trained
on un-labelled audio data, Whisper is pre-trained on a vast quantity of
**labelled** audio-transcription data, 680,000 hours to be precise.
This is an order of magnitude more data than the un-labelled audio data used
to train Wav2Vec 2.0 (60,000 hours). What is more, 117,000 hours of this
pre-training data is multilingual ASR data. This results in checkpoints
that can be applied to over 96 languages, many of which are considered
_low-resource_.

When scaled to 680,000 hours of labelled pre-training data, Whisper models
demonstrate a strong ability to generalise to many datasets and domains.
The pre-trained checkpoints achieve competitive results to state-of-the-art
ASR systems, with near 3% word error rate (WER) on the test-clean subset of
LibriSpeech ASR and a new state-of-the-art on TED-LIUM with 4.7% WER (_c.f._
Table 8 of the [Whisper paper](https://cdn.openai.com/papers/whisper.pdf)).
The extensive multilingual ASR knowledge acquired by Whisper during pre-training
can be leveraged for other low-resource languages; through fine-tuning, the
pre-trained checkpoints can be adapted for specific datasets and languages
to further improve upon these results. We'll show just how Whisper can be fine-tuned
for low-resource languages in this Colab.

<figure>
<img src="https://raw.githubusercontent.com/sanchit-gandhi/notebooks/main/whisper_architecture.svg" alt="Trulli" style="width:100%">
<figcaption align = "center"><b>Figure 1:</b> Whisper model. The architecture
follows the standard Transformer-based encoder-decoder model. A
log-Mel spectrogram is input to the encoder. The last encoder
hidden states are input to the decoder via cross-attention mechanisms. The
decoder autoregressively predicts text tokens, jointly conditional on the
encoder hidden states and previously predicted tokens. Figure source:
<a href="https://openai.com/blog/whisper/">OpenAI Whisper Blog</a>.</figcaption>
</figure>

The Whisper checkpoints come in five configurations of varying model sizes.
The smallest four are trained on either English-only or multilingual data.
The largest checkpoints are multilingual only. All 11 of the pre-trained checkpoints
are available on the [Hugging Face Hub](https://huggingface.co/models?search=openai/whisper). The
checkpoints are summarised in the following table with links to the models on the Hub:

| Size     | Layers | Width | Heads | Parameters | English-only                                         | Multilingual                                        |
|----------|--------|-------|-------|------------|------------------------------------------------------|-----------------------------------------------------|
| tiny     | 4      | 384   | 6     | 39 M       | [✓](https://huggingface.co/openai/whisper-tiny.en)   | [✓](https://huggingface.co/openai/whisper-tiny.)    |
| base     | 6      | 512   | 8     | 74 M       | [✓](https://huggingface.co/openai/whisper-base.en)   | [✓](https://huggingface.co/openai/whisper-base)     |
| small    | 12     | 768   | 12    | 244 M      | [✓](https://huggingface.co/openai/whisper-small.en)  | [✓](https://huggingface.co/openai/whisper-small)    |
| medium   | 24     | 1024  | 16    | 769 M      | [✓](https://huggingface.co/openai/whisper-medium.en) | [✓](https://huggingface.co/openai/whisper-medium)   |
| large    | 32     | 1280  | 20    | 1550 M     | x                                                    | [✓](https://huggingface.co/openai/whisper-large)    |
| large-v2 | 32     | 1280  | 20    | 1550 M     | x                                                    | [✓](https://huggingface.co/openai/whisper-large-v2) |
| large-v3 | 32     | 1280  | 20    | 1550 M     | x                                                    | [✓](https://huggingface.co/openai/whisper-large-v3) |


For demonstration purposes, we'll fine-tune the multilingual version of the
[`"small"`](https://huggingface.co/openai/whisper-small) checkpoint with 244M params (~= 1GB).
As for our data, we'll train and evaluate our system on a low-resource language
taken from the [Common Voice](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0)
dataset. We'll show that with as little as 8 hours of fine-tuning data, we can achieve
strong performance in this language.

------------------------------------------------------------------------

\\({}^1\\) The name Whisper follows from the acronym “WSPSR”, which stands for “Web-scale Supervised Pre-training for Speech Recognition”.

## Prepare Environment

First of all, let's try to secure a decent GPU for our Colab! Unfortunately, it's becoming much harder to get access to a good GPU with the free version of Google Colab. However, with Google Colab Pro one should have no issues in being allocated a V100 or P100 GPU.

To get a GPU, click _Runtime_ -> _Change runtime type_, then change _Hardware accelerator_ from _CPU_ to one of the available GPUs, e.g. _T4_ (or better if you have one available). Next, click `Connect T4` in the top right-hand corner of your screen (or `Connect {V100, A100}` if you selected a different GPU).

We can verify that we've been assigned a GPU and view its specifications:

usage: ipykernel_launcher.py [-h] --dataset_name DATASET_NAME --model_id
                             MODEL_ID --version VERSION --environment
                             {laptop,cluster,bwcluster} --train_state {T,NT}
                             [--date DATE] --device DEVICE --task
                             {classification,transcribe,joint}
                             --developer_mode {Y,N} --augmentation {Y,N}
                             --additional_tokens {Y,N}
ipykernel_launcher.py: error: the following arguments are required: --dataset_name, --model_id, --version, --environment, --train_state, --device, --task, --developer_mode, --augmentation, --additional_tokens


SystemExit: 2

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [66]:
from visualizations import plot_waveform
import pandas as pd
import os
import torchaudio
import numpy as np
"""csv_file = pd.read_csv('syntheticdata/ESC-50/meta/esc50.csv')
print(csv_file.head())
grouped_by_name =  csv_file.groupby('category')
path_noise = csv_file['filename'].iloc[0]

print(os.getcwd())
cwd = os.getcwd()

filepath = f'{cwd}/syntheticdata/ESC-50/audio/{path_noise}'
waveform,sample_rate = torchaudio.load(filepath)
waveform = waveform.numpy()

    # Play the sound using sounddevice
    #sd.play(waveform, sample_rate)
sample_rate = 44100  # 44.1 kHz
duration = 5  # 5 seconds
waveform = np.sin(2 * np.pi * 440 * np.linspace(0, duration, duration * sample_rate))

waveform = np.expand_dims(waveform, axis=0)
print(waveform.shape)
plot_waveform(waveform, sample_rate)

SyntaxError: incomplete input (2227398147.py, line 6)

In [67]:
from visualizations import plot_spec, plot_spectrogram, plot_waveform
from audioprocessing import get_spectrogram
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio
from torchaudio.utils import download_asset
import torch
import torchaudio
import numpy as np
import torchaudio.transforms as T
import pandas as pd
import os
import seaborn as sns
sns.set_theme()
SAMPLE_WAV_SPEECH_PATH = download_asset("tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav")

csv_file = pd.read_csv('syntheticdata/ESC-50/meta/esc50.csv')

path_noise = csv_file['filename'].iloc[0]

cwd = os.getcwd()

filepath = f'{cwd}/syntheticdata/ESC-50/audio/{path_noise}'
SAMPLE_WAV_SPEECH_PATH = filepath


spec = get_spectrogram(path=filepath, power=None)

spec = np.squeeze(spec)
stretch = T.TimeStretch()

spec_12 = stretch(spec, overriding_rate=1.2)
spec_09 = stretch(spec, overriding_rate=0.9)
        
waveform, sample_rate = torchaudio.load(SAMPLE_WAV_SPEECH_PATH, channels_first=False)
plot_waveform(waveform=waveform, sample_rate=sample_rate)

plot_spectrogram(spec)

  





ImportError: cannot import name 'generate_noise_dataset' from partially initialized module 'augmentations' (most likely due to a circular import) (/home/niklas/PycharmProjects/Whisper/augmentations.py)

We'll employ several popular Python packages to fine-tune the Whisper model.
We'll use `datasets[audio]` to download and prepare our training data, alongside
`transformers` and `accelerate` to load and train our Whisper model.
We'll also require the `soundfile` package to pre-process audio files,
`evaluate` and `jiwer` to assess the performance of our model, and
`tensorboard` to log our metrics. Finally, we'll use `gradio` to build a
flashy demo of our fine-tuned model.

In [ ]:
!pip install --upgrade --quiet pip
!pip install --upgrade --quiet datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio

In [ ]:
import numpy as np 
def compute_loudness(waveform, frame_size, hop_length):
    # Compute the RMS (Root Mean Square) for each frame
    waveform_squared = waveform ** 2
    loudness = torch.nn.functional.avg_pool1d(
        waveform_squared.unsqueeze(0),
        kernel_size=frame_size,
        stride=hop_length,
        padding=0
    ).squeeze(0).sqrt()

    return loudness

def compute_loudness_dB(loudness):
    # Convert RMS loudness to decibels
    # Add a small constant to avoid log10 of zero
    epsilon = 1e-6
    loudness_dB = 20 * torch.log10(loudness + epsilon)
    return loudness_dB

def plot_loudness(waveform, sample_rate, frame_size=1024, hop_length=512):
    # Compute loudness
    loudness = compute_loudness(waveform, frame_size, hop_length)
    
    # Convert loudness to decibels
    loudness_dB = compute_loudness_dB(loudness)
    
    # Generate time axis for loudness plot
    time_axis = np.linspace(0, len(waveform[0]) / sample_rate, len(loudness_dB[0]))
    
    # Plotting the loudness in dB
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, loudness_dB[0].numpy(), label="Loudness (dB)")
    plt.xlabel("Time (s)")
    plt.ylabel("Loudness (dB)")
    plt.title("Loudness in Decibels Over Time")
    plt.legend()
    plt.show()

# Load audio file
waveform, sample_rate = torchaudio.load(SAMPLE_WAV_SPEECH_PATH, channels_first=False)

# Plot the loudness in dB
#plot_loudness(waveform, sample_rate)

In [ ]:
from visualizations import plot_specgram
import torch
import torchaudio
import torchaudio.functional as F

print(torch.__version__)
print(torchaudio.__version__)

import matplotlib.pyplot as plt
# applying audio effects and overlaying the sound
# Define effects
effect = ",".join(
    [
        "lowpass=frequency=300:poles=1",  # apply single-pole lowpass filter
        "atempo=0.8",  # reduce the speed
        "aecho=in_gain=0.8:out_gain=0.9:delays=200:decays=0.3|delays=400:decays=0.3"
        # Applying echo gives some dramatic feeling
    ],
)


# Apply effects
def apply_effect(waveform, sample_rate, effect):
    effector = torchaudio.io.AudioEffector(effect=effect)
    return effector.apply(waveform, sample_rate)


waveform2 = apply_effect(waveform, sample_rate, effect)
plot_waveform(waveform2, sample_rate)
print(waveform2.shape)
print(waveform.shape)
plot_specgram(waveform2, sample_rate, title="Original", xlim=(0, 3.04))
#Audio(waveform2.T, rate=sample_rate)


In [ ]:
import torch
import torchaudio
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_3d_spectrogram(spectrogram, sample_rate):
    # Convert the spectrogram to log scale (dB)
    spectrogram_dB = 10 * torch.log10(spectrogram + 1e-10)
    
    # Create time and frequency axes
    num_frames = spectrogram.size(1)
    num_bins = spectrogram.size(0)
    
    time_axis = torch.linspace(0, num_frames / sample_rate, num_frames)
    freq_axis = torch.linspace(0, sample_rate / 2, num_bins)
    
    # Prepare the meshgrid for 3D plotting
    T, F = torch.meshgrid(time_axis, freq_axis)
    
    # Create a 3D plot
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the surface
    ax.plot_surface(T.numpy(), F.numpy(), spectrogram_dB.numpy(), cmap='viridis')
    
    # Labels and title
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Frequency [Hz]')
    ax.set_zlabel('Amplitude [dB]')
    ax.set_title('3D Spectrogram')
    
    plt.show()

# Load an audio file
waveform, sample_rate = torchaudio.load(SAMPLE_WAV_SPEECH_PATH)

# Generate a spectrogram
spectrogram_transform = torchaudio.transforms.Spectrogram(n_fft=1024, hop_length=512, power=2)
spectrogram = spectrogram_transform(waveform)

# Plot the 3D spectrogram for the first channel (if stereo)
plot_3d_spectrogram(spectrogram[0], sample_rate)
"""

We strongly advise you to upload model checkpoints directly the [Hugging Face Hub](https://huggingface.co/)
whilst training. The Hub provides:
- Integrated version control: you can be sure that no model checkpoint is lost during training.
- Tensorboard logs: track important metrics over the course of training.
- Model cards: document what a model does and its intended use cases.
- Community: an easy way to share and collaborate with the community!

Linking the notebook to the Hub is straightforward - it simply requires entering your
Hub authentication token when prompted. Find your Hub authentication token [here](https://huggingface.co/settings/tokens):

## Load Dataset

Using 🤗 Datasets, downloading and preparing data is extremely simple.
We can download and prepare the Common Voice splits in just one line of code.

First, ensure you have accepted the terms of use on the Hugging Face Hub: [mozilla-foundation/common_voice_11_0](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0). Once you have accepted the terms, you will have full access to the dataset and be able to download the data locally.

Since Hindi is very low-resource, we'll combine the `train` and `validation`
splits to give approximately 8 hours of training data. We'll use the 4 hours
of `test` data as our held-out test set:

In [75]:
from datasets import load_dataset, DatasetDict,Dataset

common_voice = DatasetDict()

common_voice["train"] = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="train+validation", use_auth_token=True)
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="test", use_auth_token=True)

print(common_voice)

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/datasets/load.py:2566: FutureWarning: 'use_auth_token' was deprecated in favor of 'token' in version 2.14.0 and will be removed in 3.0.0.
You can remove this warning by passing 'token=<use_auth_token>' instead.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 6540
    })
    test: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 2894
    })
})


Most ASR datasets only provide input audio samples (`audio`) and the
corresponding transcribed text (`sentence`). Common Voice contains additional
metadata information, such as `accent` and `locale`, which we can disregard for ASR.
Keeping the notebook as general as possible, we only consider the input audio and
transcribed text for fine-tuning, discarding the additional metadata information:

In [76]:
common_voice = common_voice.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

print(common_voice)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 6540
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2894
    })
})


In [68]:
"""import os
import pandas as pd
import torchaudio
import re
from typing import List
import glob
dipco_path = "/project/data_asr/dipco/Dipco"    
dev_path = os.path.join(dipco_path, 'audio/eval')
transcript_dev_path = os.path.join(dipco_path, 'transcriptions/eval')


def extract_prefix(file_path:str) -> str:
    pattern = r'^(.*)\.json$'
    match = re.search(pattern, file_path)
    if match:
        prefix = match.group(1)
        return prefix
    else :
        raise ValueError





def list_json_files(directory):
    # Construct the file path pattern
    pattern = os.path.join(directory, '*.json')

    # Use glob to get a list of files matching the pattern
    json_files = glob.glob(pattern)

    return json_files

def load_and_concatenate_json_files(directory):
    json_files = list_json_files(directory)

    # List to hold individual DataFrames
    data_frames = []

    for json_file in json_files:
        # Read the JSON file into a DataFrame
        df = pd.read_json(json_file)
        data_frames.append(df)

    # Concatenate all DataFrames into a single DataFrame
    combined_df = pd.concat(data_frames, ignore_index=True)

    return combined_df


df = load_and_concatenate_json_files(transcript_dev_path)

#df = pd.read_json(full_path)
transcriptions = df['words']

print(df.columns)
print(df['start_time'].head(1))
#print(full_path)

from transformers import WhisperFeatureExtractor
from typing import Dict
import pprint
import torch 
import matplotlib.pyplot as plt 
import multiprocessing
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
def expand_start_time(row):
    start_time_dict = row['start_time']
    rows = []
    for key, time_str in start_time_dict.items():
        new_row = row.copy()
        new_row['audio'] = key
        new_row['start'] = time_str
        rows.append(new_row)
    return pd.DataFrame(rows)


# Apply the function to each row and concatenate the results
expanded_df = pd.concat([expand_start_time(row) for _, row in df.iterrows()], ignore_index=True)

# Drop the original 'start_time' column
expanded_df = expanded_df.drop(columns=['start_time'])

# Function to convert time string to seconds
def time_to_seconds(time_str):
    h, m, s = map(float, time_str.split(':'))
    
    return h * 3600 + m * 60 + s

# Apply the conversion to the 'start' column
expanded_df['start'] = expanded_df['start'].apply(time_to_seconds)

def get_corresponding_end_time(dict:dict, key:str):
    end_time = [v for k,v in dict if k==key]
    return end_time
print(expanded_df.columns)
expanded_df['end'] = expanded_df.apply(lambda row: row['end_time'][row['audio']], axis=1)
expanded_df['end'] = expanded_df['end'].apply(time_to_seconds)
# removal of the end_time
expanded_df = expanded_df.drop(columns=['end_time'])
#expanded_df = expanded_df.drop(expanded_df['audio']=='close-talk')


# U01 - U05 & CH 1 - 7 

# Function to generate microphone paths
def generate_microphone_paths(row):
    paths = []
    for i in range(1, 7):
        path = f"{dev_path}/{row['session_id']}_{row['audio']}.CH{i}.wav"
        paths.append(path)

    path = f"{dev_path}/{row['session_id']}_{row['speaker_id']}.wav"
    paths.append(path)
    return paths

# Apply the function to generate the paths for each row
expanded_df['file_path'] = expanded_df.apply(generate_microphone_paths, axis=1)

# Expand the DataFrame to include the microphone paths
expanded_df = expanded_df.explode('file_path').reset_index(drop=True)


#change the seconds to frames
def get_Frames(starting_second:float, sample_rate:int, end_second:float )-> List[int] :
     return [int(starting_second*sample_rate), int(end_second*sample_rate)]

expanded_df['frames'] = expanded_df.apply(lambda row: get_Frames(row['start'], 16000, row['end']), axis=1)
expanded_df = expanded_df[expanded_df['audio'] != 'close-talk']
columns_to_drop = ['mother_tongue', 'ref', 'nativeness', 'audio', 'session_id','speaker_id', 'gender']

#get the maximum speaking duration 
expanded_df['duration'] = expanded_df.apply(lambda row: row['end'] - row['start'], axis=1)
# print(expanded_df['duration'].max()) yielded that the biggest in the dipco dataset was above 60 seconds for those an additional separation is required 
expanded_df = expanded_df.drop(columns=columns_to_drop)
# sorting for cache efficiency so far no speedup 
def validate_frames_column(frames_list):
    return len(frames_list) == 2
if expanded_df['frames'].isnull().any():
    raise ValueError("The 'frames' column contains null values.")
if not expanded_df['frames'].apply(validate_frames_column).all():
    raise ValueError("Each entry in the 'frames' column must be a list of exactly two elements [startframe, endframe].")

expanded_df[['startframe', 'endframe']] = pd.DataFrame(expanded_df['frames'].tolist(), index=expanded_df.index)

# Drop the original 'frames' column if no longer needed


print(expanded_df.shape)
pprint.pp(expanded_df.head(10))

print(expanded_df.columns)
#expanded_df['logmel'] = expanded_df.apply(lambda row: get_logmel(row['startframe'], row['endframe'], row['file_path']), axis=1)
def get_logmel(startframe: int, endframe: int, filepath: str) -> Dict[str, torch.Tensor]:
    sliced_waveform = load_audio_segment(filepath=filepath, start_frame=startframe, end_frame=endframe)
    features = feature_extractor(sliced_waveform.numpy(), sampling_rate=16000, return_tensors='pt')
    return features


    
def load_audio_segment(filepath, start_frame, end_frame):
    waveform, sample_rate = torchaudio.load(filepath)
    return waveform[:, start_frame:end_frame], sample_rate    
#print(expanded_df)
#print(expanded_df.head(10))




SyntaxError: incomplete input (3706261683.py, line 1)

In [ ]:
from datasets import Dataset as HFDataset, Features, Value, Sequence, Audio
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor
from IPython.display import Audio as au
import torchaudio 
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="English", task="transcribe")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
expanded_df = expanded_df.head(2)
print(expanded_df['file_path'][0])
torch, sample_rate = torchaudio.load(expanded_df['file_path'][0])
display(au(torch,rate=16000))
expanded_df.rename(columns={'words':'label'}, inplace = True)
dataset = Dataset.from_pandas(expanded_df)
dataset = dataset.cast_column("file_path", Audio(sampling_rate=16000))
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["file_path"]
    label = batch["label"]
    start = batch["startframe"]
    endframe = batch["endframe"]

    print(audio["array"].shape)
    temp = audio["array"][start:endframe]
    print(temp.shape)
    display(au(temp,rate=16000))
    display(au(audio["array"],rate=16000))


    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["label"]).input_ids
    return batch
#dataset = dataset.map(prepare_dataset,  num_proc=1)

In [69]:
import torchaudio
from torch.utils.data import Dataset

class AudioSegmentDataset(Dataset):
    def __init__(self, audio_paths, start_frames, end_frames, labels, sample_rate=16000):
        self.audio_paths = audio_paths
        self.start_frames = start_frames
        self.end_frames = end_frames
        self.labels = labels
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        audio_path = self.audio_paths[idx]
        start_frame = self.start_frames[idx]
        end_frame = self.end_frames[idx]
        label = self.labels[idx]
        
        # Load the audio file
        waveform, sr = torchaudio.load(audio_path, num_frames= end_frame-start_frame, frame_offset=start_frame)
        
        # Convert frame indices to sample indices
       
        
        # Slice the waveform
       
        
        return waveform, label

# Create your custom dataset
audio_dataset = AudioSegmentDataset(
    df['audio'].tolist(),
    df['start_frame'].tolist(),
    df['end_frame'].tolist(),
    df['label'].tolist()
)


NameError: name 'df' is not defined

In [ ]:
from datasets import Dataset as HFDataset, Features, Value, Sequence, Audio

# Define the features of your dataset
features = Features({
    'audio': Audio(sampling_rate=16000),
    'start_frame': Value('int64'),
    'end_frame': Value('int64'),
    'label': Value('string')
})

# Create a Hugging Face Dataset from the DataFrame
hf_dataset = HFDataset.from_pandas(expanded_df, features=features)

# Use the custom dataset class for efficient data loading
def map_to_audio_segment(example):
    audio_path = example['audio']['path']
    start_frame = example['start_frame']
    end_frame = example['end_frame']
    
    waveform, sr = torchaudio.load(audio_path)
    start_sample = int(start_frame * 16000 / sr)
    end_sample = int(end_frame * 16000 / sr)
    waveform = waveform[:, start_sample:end_sample]
    
    return {
        'audio': waveform.numpy(),
        'label': example['label']
    }

# Apply the mapping function
hf_dataset = hf_dataset.to_iterable_dataset()
hf_dataset = hf_dataset.map(map_to_audio_segment, remove_columns=['start_frame', 'end_frame'])

# Now you can use the Hugging Face Dataset with a DataLoader
from torch.utils.data import DataLoader

# Define a collate function to handle variable-length audio sequences
def collate_fn(batch):
    waveforms = [item['audio'] for item in batch]
    labels = [item['label'] for item in batch]
    return waveforms, labels

# Create a DataLoader
dataloader = DataLoader(hf_dataset, batch_size=32, collate_fn=collate_fn)

# Iterate through the DataLoader
for batch in dataloader:
    waveforms, labels = batch
    # Your training loop here
"""

## Prepare Feature Extractor, Tokenizer and Data

The ASR pipeline can be de-composed into three stages:

1. A feature extractor which pre-processes the raw audio-inputs
2. The model which performs the sequence-to-sequence mapping
3. A tokenizer which post-processes the model outputs to text format

In 🤗 Transformers, the Whisper model has an associated feature extractor and tokenizer,
called [WhisperFeatureExtractor](https://huggingface.co/docs/transformers/main/model_doc/whisper#transformers.WhisperFeatureExtractor)
and [WhisperTokenizer](https://huggingface.co/docs/transformers/main/model_doc/whisper#transformers.WhisperTokenizer)
respectively.

We'll go through details for setting-up the feature extractor and tokenizer one-by-one!

### Load WhisperFeatureExtractor

The Whisper feature extractor performs two operations:
1. Pads / truncates the audio inputs to 30s: any audio inputs shorter than 30s are padded to 30s with silence (zeros), and those longer that 30s are truncated to 30s
2. Converts the audio inputs to _log-Mel spectrogram_ input features, a visual representation of the audio and the form of the input expected by the Whisper model

<figure>
<img src="https://raw.githubusercontent.com/sanchit-gandhi/notebooks/main/spectrogram.jpg" alt="Trulli" style="width:100%">
<figcaption align = "center"><b>Figure 2:</b> Conversion of sampled audio array to log-Mel spectrogram.
Left: sampled 1-dimensional audio signal. Right: corresponding log-Mel spectrogram. Figure source:
<a href="https://ai.googleblog.com/2019/04/specaugment-new-data-augmentation.html">Google SpecAugment Blog</a>.
</figcaption>

We'll load the feature extractor from the pre-trained checkpoint with the default values:

In [77]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

### Load WhisperTokenizer

The Whisper model outputs a sequence of _token ids_. The tokenizer maps each of these token ids to their corresponding text string. For Hindi, we can load the pre-trained tokenizer and use it for fine-tuning without any further modifications. We simply have to
specify the target language and the task. These arguments inform the
tokenizer to prefix the language and task tokens to the start of encoded
label sequences:

In [78]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="English", task="transcribe")

### Combine To Create A WhisperProcessor

To simplify using the feature extractor and tokenizer, we can _wrap_
both into a single `WhisperProcessor` class. This processor object
inherits from the `WhisperFeatureExtractor` and `WhisperProcessor`,
and can be used on the audio inputs and model predictions as required.
In doing so, we only need to keep track of two objects during training:
the `processor` and the `model`:

In [79]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="English", task="transcribe")

### Prepare Data

Let's print the first example of the Common Voice dataset to see
what form the data is in:

In [80]:
print(common_voice["train"][0])
print(processor.tokenizer.is_fast)

{'audio': {'path': '/home/niklas/.cache/huggingface/datasets/downloads/extracted/372fc40ebab6043c7264d9041d5ce91ad9fbc104f1358e55fdc6af5aed6f5979/hi_train_0/common_voice_hi_26008353.mp3', 'array': array([ 5.81611368e-26, -1.48634016e-25, -9.37040538e-26, ...,
        1.06425901e-07,  4.46416450e-08,  2.61450239e-09]), 'sampling_rate': 48000}, 'sentence': 'हमने उसका जन्मदिन मनाया।'}
False


Since
our input audio is sampled at 48kHz, we need to _downsample_ it to
16kHz prior to passing it to the Whisper feature extractor, 16kHz being the sampling rate expected by the Whisper model.

We'll set the audio inputs to the correct sampling rate using dataset's
[`cast_column`](https://huggingface.co/docs/datasets/package_reference/main_classes.html?highlight=cast_column#datasets.DatasetDict.cast_column)
method. This operation does not change the audio in-place,
but rather signals to `datasets` to resample audio samples _on the fly_ the
first time that they are loaded:

In [81]:
from datasets import Audio

train_subset = common_voice['train'].select(range(100))
test_subset = common_voice['test'].select(range(100))

# Create a new DatasetDict with the subsets
common_voice = DatasetDict({
    'train': train_subset,
    'test': test_subset
})

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

Re-loading the first audio sample in the Common Voice dataset will resample
it to the desired sampling rate:

In [82]:
print(common_voice["train"][0])

{'audio': {'path': '/home/niklas/.cache/huggingface/datasets/downloads/extracted/372fc40ebab6043c7264d9041d5ce91ad9fbc104f1358e55fdc6af5aed6f5979/hi_train_0/common_voice_hi_26008353.mp3', 'array': array([ 3.81639165e-17,  2.42861287e-17, -1.73472348e-17, ...,
       -1.30981789e-07,  2.63096808e-07,  4.77157300e-08]), 'sampling_rate': 16000}, 'sentence': 'हमने उसका जन्मदिन मनाया।'}


Now we can write a function to prepare our data ready for the model:
1. We load and resample the audio data by calling `batch["audio"]`. As explained above, 🤗 Datasets performs any necessary resampling operations on the fly.
2. We use the feature extractor to compute the log-Mel spectrogram input features from our 1-dimensional audio array.
3. We encode the transcriptions to label ids through the use of the tokenizer.

In [83]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

We can apply the data preparation function to all of our training examples using dataset's `.map` method. The argument `num_proc` specifies how many CPU cores to use. Setting `num_proc` > 1 will enable multiprocessing. If the `.map` method hangs with multiprocessing, set `num_proc=1` and process the dataset sequentially.

In [84]:
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], num_proc=1)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [85]:
processor.tokenizer.is_fast

False

## Training and Evaluation

Now that we've prepared our data, we're ready to dive into the training pipeline.
The [🤗 Trainer](https://huggingface.co/transformers/master/main_classes/trainer.html?highlight=trainer)
will do much of the heavy lifting for us. All we have to do is:

- Load a pre-trained checkpoint: we need to load a pre-trained checkpoint and configure it correctly for training.

- Define a data collator: the data collator takes our pre-processed data and prepares PyTorch tensors ready for the model.

- Evaluation metrics: during evaluation, we want to evaluate the model using the [word error rate (WER)](https://huggingface.co/metrics/wer) metric. We need to define a `compute_metrics` function that handles this computation.

- Define the training configuration: this will be used by the 🤗 Trainer to define the training schedule.

Once we've fine-tuned the model, we will evaluate it on the test data to verify that we have correctly trained it
to transcribe speech in Hindi.

### Load a Pre-Trained Checkpoint

We'll start our fine-tuning run from the pre-trained Whisper `small` checkpoint,
the weights for which we need to load from the Hugging Face Hub. Again, this
is trivial through use of 🤗 Transformers!

In [86]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

We can disable the automatic language detection task performed during inference, and force the model to generate in Hindi. To do so, we set the [langauge](https://huggingface.co/docs/transformers/en/model_doc/whisper#transformers.WhisperForConditionalGeneration.generate.language)
and [task](https://huggingface.co/docs/transformers/en/model_doc/whisper#transformers.WhisperForConditionalGeneration.generate.task)
arguments to the generation config. We'll also set any [`forced_decoder_ids`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate.forced_decoder_ids)
to None, since this was the legacy way of setting the language and
task arguments:

In [87]:
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

### Define a Data Collator

The data collator for a sequence-to-sequence speech model is unique in the sense that it
treats the `input_features` and `labels` independently: the  `input_features` must be
handled by the feature extractor and the `labels` by the tokenizer.

The `input_features` are already padded to 30s and converted to a log-Mel spectrogram
of fixed dimension by action of the feature extractor, so all we have to do is convert the `input_features`
to batched PyTorch tensors. We do this using the feature extractor's `.pad` method with `return_tensors=pt`.

The `labels` on the other hand are un-padded. We first pad the sequences
to the maximum length in the batch using the tokenizer's `.pad` method. The padding tokens
are then replaced by `-100` so that these tokens are **not** taken into account when
computing the loss. We then cut the BOS token from the start of the label sequence as we
append it later during training.

We can leverage the `WhisperProcessor` we defined earlier to perform both the
feature extractor and the tokenizer operations:

In [88]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

Let's initialise the data collator we've just defined:

In [89]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

### Evaluation Metrics

We'll use the word error rate (WER) metric, the 'de-facto' metric for assessing
ASR systems. For more information, refer to the WER [docs](https://huggingface.co/metrics/wer). We'll load the WER metric from 🤗 Evaluate:

In [90]:
import evaluate

metric = evaluate.load("wer")

We then simply have to define a function that takes our model
predictions and returns the WER metric. This function, called
`compute_metrics`, first replaces `-100` with the `pad_token_id`
in the `label_ids` (undoing the step we applied in the
data collator to ignore padded tokens correctly in the loss).
It then decodes the predicted and label ids to strings. Finally,
it computes the WER between the predictions and reference labels:

In [91]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

### Define the Training Configuration

In the final step, we define all the parameters related to training. For more detail on the training arguments, refer to the Seq2SeqTrainingArguments [docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments).

In [92]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-hi",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=0,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=10,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


**Note**: if one does not want to upload the model checkpoints to the Hub,
set `push_to_hub=False`.

We can forward the training arguments to the 🤗 Trainer along with our model,
dataset, data collator and `compute_metrics` function:

In [93]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


We'll save the processor object once before starting training. Since the processor is not trainable, it won't change over the course of training:

In [94]:
processor.save_pretrained(training_args.output_dir)

[]

In [95]:
import transformers
print(transformers.__version__)

4.44.0


### Training

Training will take approximately 5-10 hours depending on your GPU or the one
allocated to this Google Colab. If using this Google Colab directly to
fine-tune a Whisper model, you should make sure that training isn't
interrupted due to inactivity. A simple workaround to prevent this is
to paste the following code into the console of this tab (_right mouse click_
-> _inspect_ -> _Console tab_ -> _insert code_).

```javascript
function ConnectButton(){
    console.log("Connect pushed");
    document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click()
}
setInterval(ConnectButton, 60000);
```

The peak GPU memory for the given training configuration is approximately 15.8GB.
Depending on the GPU allocated to the Google Colab, it is possible that you will encounter a CUDA `"out-of-memory"` error when you launch training.
In this case, you can reduce the `per_device_train_batch_size` incrementally by factors of 2
and employ [`gradient_accumulation_steps`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments.gradient_accumulation_steps)
to compensate.

To launch training, simply execute:

In [96]:

trainer.train()

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss,Wer
10,No log,1.983261,281.544029
20,No log,1.674430,188.057901


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

Our best WER is 32.0% - not bad for 8h of training data! We can make our model more accessible on the Hub with appropriate tags and README information.
You can change these values to match your dataset, language and model
name accordingly:

In [ ]:
kwargs = {
    "dataset_tags": "mozilla-foundation/common_voice_11_0",
    "dataset": "Common Voice 11.0",  # a 'pretty' name for the training dataset
    "dataset_args": "config: hi, split: test",
    "language": "hi",
    "model_name": "Whisper Small Hi - Sanchit Gandhi",  # a 'pretty' name for our model
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
}

The training results can now be uploaded to the Hub. To do so, execute the `push_to_hub` command and save the preprocessor object we created:

## Building a Demo

Now that we've fine-tuned our model we can build a demo to show
off its ASR capabilities! We'll make use of 🤗 Transformers
`pipeline`, which will take care of the entire ASR pipeline,
right from pre-processing the audio inputs to decoding the
model predictions.

Running the example below will generate a Gradio demo where we
can record speech through the microphone of our computer and input it to
our fine-tuned Whisper model to transcribe the corresponding text:

In [ ]:
'''from transformers import pipeline
import gradio as gr

pipe = pipeline(model="sanchit-gandhi/whisper-small-hi")  # change to "your-username/the-name-you-picked"

def transcribe(audio):
    text = pipe(audio)["text"]
    return text

iface = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(source="microphone", type="filepath"),
    outputs="text",
    title="Whisper Small Hindi",
    description="Realtime demo for Hindi speech recognition using a fine-tuned Whisper small model.",
)

iface.launch()'''

In [ ]:
"""
A simple walkthrough of how to code a convolutional neural network (CNN)
using the PyTorch library. For demonstration we train it on the very
common MNIST dataset of handwritten digits. In this code we go through
how to create the network as well as initialize a loss function, optimizer,
check accuracy and more.

Programmed by Aladdin Persson
* 2020-04-08: Initial coding
* 2021-03-24: More detailed comments and small revision of the code
* 2022-12-19: Small revision of code, checked that it works with latest PyTorch version

"""

# Imports
import torch
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import (
    DataLoader,
)  # Gives easier dataset managment by creating mini batches etc.
from tqdm import tqdm  # For nice progress bar!

# Simple CNN
class CNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=8,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        self.fc1 = nn.Linear(16 * 7 * 7, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        return x


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
in_channels = 1
num_classes = 10
learning_rate = 3e-4 # karpathy's constant
batch_size = 64
num_epochs = 3

# Load Data
train_dataset = datasets.MNIST(
    root="dataset/", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.MNIST(
    root="dataset/", train=False, transform=transforms.ToTensor(), download=True
)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

# Initialize network
model = CNN(in_channels=in_channels, num_classes=num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Train Network
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)

        # forward
        scores = model(data)
        loss = criterion(scores, targets)

        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()

# Check accuracy on training & test to see how good our model
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()
    return num_correct / num_samples


print(f"Accuracy on training set: {check_accuracy(train_loader, model)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model)*100:.2f}")

In [ ]:
ds = DatasetDict({
"train": Dataset.from_pandas(df_train.reset_index(drop=True)),
"valid": Dataset.from_pandas(df_valid.reset_index(drop=True)),
"test": Dataset.from_pandas(df_test.reset_index(drop=True)),
"unsup": Dataset.from_pandas(df_unsup.reset_index(drop=True))})

## Closing Remarks

In this blog, we covered a step-by-step guide on fine-tuning Whisper for multilingual ASR
using 🤗 Datasets, Transformers and the Hugging Face Hub. For more details on the Whisper model, the Common Voice dataset and the theory behind fine-tuning, refere to the accompanying [blog post](https://huggingface.co/blog/fine-tune-whisper). If you're interested in fine-tuning other
Transformers models, both for English and multilingual ASR, be sure to check out the
examples scripts at [examples/pytorch/speech-recognition](https://github.com/huggingface/transformers/tree/main/examples/pytorch/speech-recognition).

In [2]:
import os
import sqlite3
import shutil
import platform
from datetime import datetime

def get_firefox_profile_path():
    # Detect the operating system
    system = platform.system()

    if system == 'Linux':
        firefox_profile_dir = os.path.expanduser('~/.mozilla/firefox')
    elif system == 'Darwin':  # macOS
        firefox_profile_dir = os.path.expanduser('~/Library/Application Support/Firefox')
    elif system == 'Windows':
        firefox_profile_dir = os.path.expanduser('~\\AppData\\Roaming\\Mozilla\\Firefox')
    else:
        print("Unsupported operating system.")
        return None

    profiles_ini = os.path.join(firefox_profile_dir, 'profiles.ini')

    if not os.path.exists(profiles_ini):
        print("Firefox profile not found.")
        return None

    with open(profiles_ini, 'r') as f:
        lines = f.readlines()

    for line in lines:
        if line.strip().startswith('Path='):
            profile_path = line.split('=')[1].strip()
            # Check if it's the default profile
            if "default" in profile_path:
                return os.path.join(firefox_profile_dir, profile_path)
    
    # Fall back to the first profile if "default" is not found
    for line in lines:
        if line.strip().startswith('Path='):
            profile_path = line.split('=')[1].strip()
            return os.path.join(firefox_profile_dir, profile_path)

    print("No Firefox profile detected.")
    return None

def extract_firefox_history(output_file):
    profile_path = get_firefox_profile_path()
    if not profile_path:
        print("No Firefox profile detected.")
        return

    places_db_path = os.path.join(profile_path, 'places.sqlite')

    # Check if the places.sqlite file exists
    if not os.path.exists(places_db_path):
        print("No places.sqlite file found in the profile.")
        return

    # Copy the database to avoid locking issues
    temp_db_path = '/tmp/places.sqlite'
    shutil.copy2(places_db_path, temp_db_path)

    # Connect to the copied database
    conn = sqlite3.connect(temp_db_path)
    cursor = conn.cursor()

    # Query the history table
    cursor.execute('''
        SELECT url, datetime((visit_date/1000000), 'unixepoch', 'localtime') AS visit_date
        FROM moz_places
        JOIN moz_historyvisits ON moz_places.id = moz_historyvisits.place_id
        ORDER BY visit_date DESC
    ''')

    rows = cursor.fetchall()

    # Write the history to the output file
    with open(output_file, 'w') as f:
        for row in rows:
            url, visit_date = row
            f.write(f"{visit_date} - {url}\n")

    # Cleanup
    conn.close()
    os.remove(temp_db_path)

if __name__ == '__main__':
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_file = f"firefox_history_snapshot_{timestamp}.txt"
    extract_firefox_history(output_file)
    print(f"Firefox history saved to {output_file}")


Firefox profile not found.
No Firefox profile detected.
Firefox history saved to firefox_history_snapshot_20241009_172822.txt
